In [1]:
import requests
from bs4 import BeautifulSoup
from IPython.core.display import HTML
import pandas as pd
import io

In [2]:
URL = 'https://www.countrymusichalloffame.org/hall-of-fame'

response = requests.get(URL)

In [3]:
response.status_code

200

In [4]:
soup = BeautifulSoup(response.text)

In [5]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <!-- be_ixf, sdk, gho-->
  <meta content="php_sdk_1.5.12" name="be:sdk"/>
  <meta content="98ms" name="be:timer"/>
  <meta content="https%3A%2F%2Fwww.countrymusichalloffame.org%2Fhall-of-fame" name="be:orig_url"/>
  <meta content="https%3A%2F%2Fwww.countrymusichalloffame.org%2Fhall-of-fame" name="be:norm_url"/>
  <meta content="https%3A%2F%2Fixfd1-api.bc0a.com%2Fapi%2Fixf%2F1.0.0%2Fget_capsule%2Ff00000000317463%2F0602379402" name="be:capsule_url"/>
  <meta content="py_2025;pm_09;pd_13;ph_17;pmh_27;p_epoch:1757784435344" name="be:api_dt"/>
  <meta content="py_2025;pm_09;pd_13;ph_17;pmh_27;p_epoch:1757784435344" name="be:mod_dt"/>
  <meta content="Qkdy8Cr6nhjy2N5bPZC0dEHL30YgRwf6EXEqgGZ89MBlmTKB/283KAhoZG7Gyrp6MgZwEcmLupuFJl1Z+3+McBuHwRv2j3cq4chr7385lMr6+foZ9MP9XBpdnfQl6qz96oBs15s7WOeeZP7c56fsbyQD52nPTkwmNjHP6XOkVYLnACuA5woBdNVDCF04Dlc3asnQnvZasWg2BQ5GD9K6ehaRZosHAazOel+0bKwpyl1undDRgrOHNcbVRUqjeUQaOmp+9nE8GJL4FmonQ3lR/fJC+ArXyR8+u9xw8MvNNMqH3on

#### 1.	Start by using either the inspector or by viewing the page source. Can you identify a tag that might be helpful for finding the names of all inductees? Make use of this to create a list containing just the names of each inductee.


In [7]:
soup.find('h2').text

'\n                    Roy Acuff                '

In [8]:
inductee_names = [name.text.strip() for name in soup.find_all('h2')]

print(f"Amount of inductee names: {len(inductee_names)}")

Amount of inductee names: 310


In [9]:
print(inductee_names[:5])

['Roy Acuff', 'Alabama', 'Bill Anderson', 'John Anderson', 'Eddy Arnold']


#### 2.	Next, try and find a tag that could be used to find the year that each member was inducted. Extract these into a list. When you do this, be sure to only include the year and not the full text. For example, for Roy Acuff, the list entry should be "1962" and not "Inducted 1962". Double-check that the resulting list has the correct number of elements and is in the same order as your inductees list.

In [11]:
soup.find('h3').text

'\n                    Inducted 1962                                    '

In [12]:
inductee_year = [year.text.strip().split()[-1] for year in soup.find_all('h3')]

print(f"Amount of induxtee years: {len(inductee_year)}")

Amount of induxtee years: 310


In [13]:
print(inductee_year[:5])

['1962', '2005', '2001', '2024', '1966']


#### 3.	Take the two lists you created on parts 1 and 2 and convert it into a pandas DataFrame.


In [15]:
df = pd.DataFrame({
    "Name": inductee_names,
    "Year": inductee_year
})

print(df)

               Name  Year
0         Roy Acuff  1962
1           Alabama  2005
2     Bill Anderson  2001
3     John Anderson  2024
4       Eddy Arnold  1966
..              ...   ...
305      Bob McDill  2023
306  Patty Loveless  2023
307    James Burton  2024
308   John Anderson  2024
309      Toby Keith  2024

[310 rows x 2 columns]


#### 4.	If you navigate to Roy Acuff's page, you will see that his date of birth and date of death are listed towards the top of the page, along with his birthplace. Write some code that will extract these three values. Once you get it working for Roy Acuff, figure out how you can extract these values across the whole dataset of artists. In doing this, you'll need to figure out a way to automatically determine the correct urls for each artist. Note also that not every artist will have these three values, so write your code in a way that it can handle cases where these values are missing. Alabama is one such example.

In [17]:
person_urls = ["https://www.countrymusichalloffame.org/hall-of-fame/" + name.replace(" ", "-") for name in inductee_names]

print(person_urls[:5])

['https://www.countrymusichalloffame.org/hall-of-fame/Roy-Acuff', 'https://www.countrymusichalloffame.org/hall-of-fame/Alabama', 'https://www.countrymusichalloffame.org/hall-of-fame/Bill-Anderson', 'https://www.countrymusichalloffame.org/hall-of-fame/John-Anderson', 'https://www.countrymusichalloffame.org/hall-of-fame/Eddy-Arnold']


In [18]:
df['person_url'] = person_urls

df.head(5)

,Name,Year,person_url
0,Roy Acuff,1962,https://www.countrymusichalloffame.org/hall-of...
1,Alabama,2005,https://www.countrymusichalloffame.org/hall-of...
2,Bill Anderson,2001,https://www.countrymusichalloffame.org/hall-of...
3,John Anderson,2024,https://www.countrymusichalloffame.org/hall-of...
4,Eddy Arnold,1966,https://www.countrymusichalloffame.org/hall-of...


In [19]:
def get_date_of_birth(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text)

        date_of_birth_block = soup.find("span", class_="headerPerson__details-item--term", string=lambda s: s and "Born" in s)
        
        date_of_birth = (date_of_birth_block.find_next("div", class_="headerPerson__details-item--def").text.strip())
        
        return date_of_birth
    except:
        # print(f"Error for {url}")
        return "unknown"

In [20]:
def get_date_of_death(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text)

        date_of_death_block = soup.find("span", class_="headerPerson__details-item--term", string=lambda s: s and "Died" in s)
        
        date_of_death = (date_of_death_block.find_next("div", class_="headerPerson__details-item--def").text.strip())
        
        return date_of_death
    except:
        # print(f"Error for {url}")
        return "unknown"

In [21]:
def get_birthplace(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text)

        birthplace_block = soup.find("span", class_="headerPerson__details-item--term", string=lambda s: s and "Birthplace" in s)
        
        birthplace = (birthplace_block.find_next("div", class_="headerPerson__details-item--def").text.strip())
        
        return birthplace
    except:
        # print(f"Error for {url}")
        return "unknown"

In [22]:
df['date_of_birth'] = df['person_url'].apply(get_date_of_birth)
df['date_of_death'] = df['person_url'].apply(get_date_of_death)
df['birthplace'] = df['person_url'].apply(get_birthplace)

df.head()

,Name,Year,person_url,date_of_birth,date_of_death,birthplace
0,Roy Acuff,1962,https://www.countrymusichalloffame.org/hall-of...,"September 15, 1903","November 23, 1992","Maynardville, Tennessee"
1,Alabama,2005,https://www.countrymusichalloffame.org/hall-of...,unknown,unknown,unknown
2,Bill Anderson,2001,https://www.countrymusichalloffame.org/hall-of...,"November 1, 1937",unknown,"Columbia, South Carolina"
3,John Anderson,2024,https://www.countrymusichalloffame.org/hall-of...,"December 13, 1954",unknown,"Orlando, Florida"
4,Eddy Arnold,1966,https://www.countrymusichalloffame.org/hall-of...,"May 15, 1918","May 8, 2008","Henderson, Tennessee"
